# SANADA Target Speaker Extraction (USEF-TFGridNet)
## Google Colab Pro+ Multi-Stage Training & Evaluation Pipeline
### Guide for Professor / Research Advisor (NVIDIA L4 GPU)

This notebook runs **full-scale training** of the official **USEF-TFGridNet** (15.64M parameters) on the **SANADA acoustic dataset**.

---

### Key Capabilities & Enhancements Implemented:
1. **Multi-Speaker Mixture Scaling**:
   - **Stage A**: 2-speaker extraction (1,000 mixtures $\to$ 2,000 training pairs)
   - **Stage B**: 4-speaker extraction (1,000 mixtures $\to$ 4,000 training pairs)
   - **Stage C**: Curriculum mixture scaling (15,000 combined pairs across 2, 4, and 9 speakers)
2. **Improved Reference Conditioning**:
   - Multi-utterance enrollment pooling (`--num-enroll-samples 2`)
   - Energy-based Voice Activity Detection (`--use-vad`) to trim silent frames
3. **Composite Loss Optimization**:
   - True Scale-Invariant Signal-to-Distortion Ratio (**SI-SDR**)
   - Multi-Resolution STFT spectral convergence loss ($eta = 0.5$)
   - Cross-talk interferer leakage penalty ($\gamma = 0.1$)
4. **Reliability & Cloud Persistence**:
   - All checkpoints and metrics persist directly in Google Drive
   - Full checkpoint resumption (`--resume`) supported if session times out


### Step 1: Verify Hardware Acceleration (NVIDIA L4 GPU)
Check GPU allocation, memory, and compute capability. Colab Pro+ typically provides an **NVIDIA L4 GPU (24 GB VRAM)**.


In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Active GPU:     {props.name}")
    print(f"Total VRAM:     {props.total_memory / 1e9:.2f} GB")
    print(f"Compute Cap:    {props.major}.{props.minor}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> L4 GPU.")


Fri Nov 20 04:28:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   27C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Mount Google Drive
Mount Google Drive so that trained checkpoints, training logs, and loss curves persist safely.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Step 3: Install Required Audio & Scientific Libraries


In [ ]:
!pip install -q soundfile torchaudio pandas scipy matplotlib


### Step 4: Setup Project Codebase
Unpack the project code (`SANADA_TSE_Project.zip`) into `/content/SANADA_final_Project`.


In [ ]:

import os
import sys
import zipfile
from pathlib import Path

# Exact location of your ZIP file
ZIP_PATH = "/content/drive/MyDrive/USEF-TSE/SANADA_TSE_Project.zip"

# Destination folder in Colab
PROJECT_ROOT = Path("/content/SANADA_final_Project")

# Verify ZIP exists
if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"ZIP not found: {ZIP_PATH}")

# Create destination and extract
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

print("Extracting:", ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(PROJECT_ROOT)

print("Extraction completed!")

# Find project files
for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(str(PROJECT_ROOT), "").count(os.sep)
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")
        for f in files:
            print("  " * (level + 1) + f)

# Locate src directory
src_dirs = list(PROJECT_ROOT.rglob("src"))

valid_src = [
    p for p in src_dirs
    if (p / "model.py").exists()
    and (p / "dataset.py").exists()
    and (p / "losses.py").exists()
]

if not valid_src:
    raise FileNotFoundError(
        "Could not find src containing model.py, dataset.py and losses.py"
    )

SRC_DIR = valid_src[0]

print("\nProject source found:", SRC_DIR)

os.chdir(SRC_DIR.parent)
sys.path.insert(0, str(SRC_DIR))

print("Working directory:", os.getcwd())
print("Source directory:", SRC_DIR)


Extracting: /content/drive/MyDrive/USEF-TSE/SANADA_TSE_Project.zip
Extraction completed!
SANADA_final_Project/
  requirements.txt
  SANADA_TSE_Training.ipynb
  SANADA_USEF_Colab_Training.ipynb
  README.md
  CLUSTER_TRAINING_GUIDE.md
  pretrained_model/
    usef_tfgridnet_wsj02mix.pth.tar
    config.yaml
  experiments/
    fine_tuned/
      best_sanada_usef.pt
  src/
    feature.py
    losses.py
    final_env_verify.py
    dataset.py
    independent_verify.py
    verify_reproducibility.py
    model.py
    evaluate.py
    train.py
    inference.py
    strict_functional_test.py
    models/
      model_USEF_TFGridNet.py
      model_USEF_SepFormer.py

Project source found: /content/SANADA_final_Project/src
Working directory: /content/SANADA_final_Project
Source directory: /content/SANADA_final_Project/src


### Step 5: Configure Paths for SANADA Dataset & Google Drive Outputs

> **Important Configuration for Sir**:
> 1. Set `DRIVE_DATASET_DIR` to the folder where you have stored your `SANADA/` dataset in Google Drive.
> 2. Set `DRIVE_EXPERIMENTS_DIR` to where you want trained checkpoints saved in your Google Drive.


In [ ]:
import os
from pathlib import Path

# ==============================================================================
# CONFIGURATION: Please set your Google Drive paths below
# ==============================================================================
# 1. Path to your SANADA dataset folder (or parent folder containing 'SANADA/'):
DRIVE_DATASET_DIR = '/content/drive/MyDrive/SANADA'  # <-- ADJUST IF NEEDED

# 2. Path in Google Drive to store all checkpoints and logs:
DRIVE_EXPERIMENTS_DIR = '/content/drive/MyDrive/SANADA_USEF_Experiments'
os.makedirs(DRIVE_EXPERIMENTS_DIR, exist_ok=True)

# 3. Path to official pretrained WSJ0-2mix weights:
# (Checks local project copy first, then Google Drive)
LOCAL_PRETRAINED = Path('/content/SANADA_final_Project/pretrained_model/usef_tfgridnet_wsj02mix.pth.tar')
DRIVE_PRETRAINED = Path('/content/drive/MyDrive/pretrained_model/usef_tfgridnet_wsj02mix.pth.tar')

if LOCAL_PRETRAINED.exists():
    INIT_CKPT = str(LOCAL_PRETRAINED)
elif DRIVE_PRETRAINED.exists():
    INIT_CKPT = str(DRIVE_PRETRAINED)
elif os.path.exists('/content/drive/MyDrive/best_sanada_usef.pt'):
    INIT_CKPT = '/content/drive/MyDrive/best_sanada_usef.pt'
else:
    INIT_CKPT = str(LOCAL_PRETRAINED)

print("=" * 70)
print("PATH VERIFICATION:")
print("  Dataset Directory:     ", DRIVE_DATASET_DIR)
print("  Google Drive Save Dir: ", DRIVE_EXPERIMENTS_DIR)
print("  Init Checkpoint:       ", INIT_CKPT)
print("=" * 70)

# Automatic validation check
ds_p = Path(DRIVE_DATASET_DIR)
test_csv_1 = ds_p / "SANADA" / "Org_Fsd_Mix" / "Org_Mix_2_Sounds" / "train" / "metadata.csv"
test_csv_2 = ds_p / "Org_Fsd_Mix" / "Org_Mix_2_Sounds" / "train" / "metadata.csv"

if test_csv_1.exists():
    print("SUCCESS: SANADA dataset structure detected at:", test_csv_1.parent)
elif test_csv_2.exists():
    print("SUCCESS: SANADA dataset structure detected at:", test_csv_2.parent)
else:
    print("NOTE: Please verify that DRIVE_DATASET_DIR points to your SANADA folder.")


PATH VERIFICATION:
  Dataset Directory:      /content/drive/MyDrive/SANADA
  Google Drive Save Dir:  /content/drive/MyDrive/SANADA_USEF_Experiments
  Init Checkpoint:        /content/SANADA_final_Project/pretrained_model/usef_tfgridnet_wsj02mix.pth.tar
SUCCESS: SANADA dataset structure detected at: /content/drive/MyDrive/SANADA/Org_Fsd_Mix/Org_Mix_2_Sounds/train


### Step 6: Quick 1-Step CUDA Sanity Check
Runs a single forward and backward step on the L4 GPU to verify all imports, tensor dimensions, and memory usage before launching training.


In [ ]:
import os
import sys

print("Current directory:", os.getcwd())
print("Files here:", os.listdir())
print("Does src exist?", os.path.isdir("src"))

if os.path.isdir("src"):
    print("Files inside src:", os.listdir("src"))


Current directory: /content/SANADA_final_Project
Files here: ['pretrained_model', 'experiments', 'requirements.txt', 'src', 'SANADA_TSE_Training.ipynb', 'SANADA_USEF_Colab_Training.ipynb', 'README.md', 'CLUSTER_TRAINING_GUIDE.md']
Does src exist? True
Files inside src: ['feature.py', 'losses.py', 'final_env_verify.py', 'dataset.py', 'independent_verify.py', 'verify_reproducibility.py', 'model.py', 'evaluate.py', 'train.py', 'models', 'inference.py', 'strict_functional_test.py']


In [ ]:

import sys
import torch

sys.path.insert(0, 'src')

from dataset import build_dataset
from model import build_usef_tfgridnet
from losses import build_loss_fn

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Running CUDA Sanity Check on:', device)

# 1. Model initialization
model = build_usef_tfgridnet(
    device=device,
    use_gradient_checkpointing=True
)
model.train()

# 2. Loss initialization
loss_fn = build_loss_fn(
    loss_type='composite',
    stft_weight=0.5,
    leakage_weight=0.1
).to(device)

# 3. Forward pass with mock batch
mix = torch.randn(1, 32000, device=device)
ref = torch.randn(1, 24000, device=device)
tgt = torch.randn(1, 32000, device=device)
interf = mix - tgt

# 4. Forward and backward pass
est = model(mix, ref)

loss_dict = loss_fn(
    est,
    tgt,
    interferer=interf
)

loss_dict['loss'].backward()

# 5. Report results
print('Sanity Check PASSED!')
print('Loss:', loss_dict['loss'].item())

if device == 'cuda':
    print(
        'Peak VRAM:',
        torch.cuda.max_memory_allocated(0) / 1e9,
        'GB'
    )

Running CUDA Sanity Check on: cuda
Sanity Check PASSED!
Loss: 46.91961669921875
Peak VRAM: 2.291723264 GB


---
## Stage A: 2-Speaker Mixture Training (`Org_Mix_2_Sounds`)

- **Dataset**: 1,000 mixture files $\to$ **2,000 speaker extraction pairs**
- **Epochs**: 10
- **Batch Size**: 4 (effective batch size = 8 with accum-steps = 2 on NVIDIA L4)
- **Loss**: Composite ($	ext{SI-SDR} + 0.5 \cdot 	ext{MR-STFT} + 0.1 \cdot 	ext{Leakage}$)
- **Conditioning**: Energy VAD enabled, 2 pooled enrollment utterances per speaker
- **Output Directory**: `$DRIVE_EXPERIMENTS_DIR/stage_A_2spk`


In [ ]:
STAGE_A_SAVE_DIR = f"{DRIVE_EXPERIMENTS_DIR}/stage_A_2spk"
os.makedirs(STAGE_A_SAVE_DIR, exist_ok=True)

!python src/train.py \
    --mix-type Org_Mix_2_Sounds \
    --epochs 10 \
    --lr 5e-5 \
    --batch-size 4 \
    --accum-steps 2 \
    --num-workers 2 \
    --device cuda \
    --datasets-dir "$DRIVE_DATASET_DIR" \
    --init-checkpoint "$INIT_CKPT" \
    --save-dir "$STAGE_A_SAVE_DIR" \
    --loss-type composite \
    --stft-loss-weight 0.5 \
    --leakage-loss-weight 0.1 \
    --num-enroll-samples 2 \
    --use-vad


FINE-TUNING OFFICIAL PRETRAINED USEF-TFGRIDNET ON SANADA
Device:             cuda
Checkpoint Source:  usef_tfgridnet_wsj02mix.pth.tar
Resume Mode:        False
Epochs:             10
Base Learning Rate: 5e-05
Batch Size:         4 (Effective batch size: 8)
DataLoader Workers: 2
Mix Type:           Org_Mix_2_Sounds
Train Samples Cap:  Full dataset
Val Samples Cap:    Full dataset
Loss Objective:     COMPOSITE (STFT Weight: 0.5, Leakage Weight: 0.1)
Enrollment Mode:    2 utterance(s) (VAD: ENABLED)
[Model] Initializing from checkpoint: usef_tfgridnet_wsj02mix.pth.tar
[Model] Architecture loaded successfully (15,641,885 parameters)
[Memory Optimization] Enabled Gradient Checkpointing (Peak VRAM: ~2.1 GB on 4s chunk)
[Dataset] Looking for SANADA dataset at: /content/drive/.shortcut-targets-by-id/1TxhAg2JEd9vvBxHppDbfGylpUZbW3x7z/SANADA
[Dataset] Mix type: Org_Mix_2_Sounds
[Dataset] Train samples: 2,000 | Validation samples: 200

[Validation Baseline] Evaluating model on validation set befo

#### *Optional: Resume Stage A if Colab Disconnected*
Uncomment and run this cell to continue Stage A training from the latest checkpoint without losing progress:


In [ ]:
# !python src/train.py \
#     --mix-type Org_Mix_2_Sounds \
#     --epochs 10 \
#     --lr 5e-5 \
#     --batch-size 4 \
#     --accum-steps 2 \
#     --num-workers 2 \
#     --device cuda \
#     --datasets-dir "$DRIVE_DATASET_DIR" \
#     --init-checkpoint "$STAGE_A_SAVE_DIR/latest_checkpoint.pt" \
#     --save-dir "$STAGE_A_SAVE_DIR" \
#     --loss-type composite \
#     --stft-loss-weight 0.5 \
#     --leakage-loss-weight 0.1 \
#     --num-enroll-samples 2 \
#     --use-vad \
#     --resume


---
## Stage B: 4-Speaker Mixture Training (`Org_Mix_4_Sounds`)

- **Dataset**: 1,000 mixture files $\to$ **4,000 speaker extraction pairs**
- **Epochs**: 5
- **Learning Rate**: $3 \times 10^{-5}$
- **Initialization**: Automatically loads `$STAGE_A_SAVE_DIR/best_sanada_usef.pt`
- **Output Directory**: `$DRIVE_EXPERIMENTS_DIR/stage_B_4spk`


In [ ]:
STAGE_B_SAVE_DIR = f"{DRIVE_EXPERIMENTS_DIR}/stage_B_4spk"
STAGE_A_BEST = f"{STAGE_A_SAVE_DIR}/best_sanada_usef.pt"
os.makedirs(STAGE_B_SAVE_DIR, exist_ok=True)

if not os.path.exists(STAGE_A_BEST):
    print(f"WARNING: Stage A checkpoint not found at: {STAGE_A_BEST}")
    print("Please complete Stage A first, or point STAGE_A_BEST to an existing checkpoint.")
else:
    print(f"Initializing Stage B from Stage A best: {STAGE_A_BEST}")

!python src/train.py \
    --mix-type Org_Mix_4_Sounds \
    --epochs 5 \
    --lr 3e-5 \
    --batch-size 4 \
    --accum-steps 2 \
    --num-workers 2 \
    --device cuda \
    --datasets-dir "$DRIVE_DATASET_DIR" \
    --init-checkpoint "$STAGE_A_BEST" \
    --save-dir "$STAGE_B_SAVE_DIR" \
    --loss-type composite \
    --stft-loss-weight 0.5 \
    --leakage-loss-weight 0.1 \
    --num-enroll-samples 2 \
    --use-vad


Initializing Stage B from Stage A best: /content/drive/MyDrive/SANADA_USEF_Experiments/stage_A_2spk/best_sanada_usef.pt
FINE-TUNING OFFICIAL PRETRAINED USEF-TFGRIDNET ON SANADA
Device:             cuda
Checkpoint Source:  best_sanada_usef.pt
Resume Mode:        False
Epochs:             5
Base Learning Rate: 3e-05
Batch Size:         4 (Effective batch size: 8)
DataLoader Workers: 2
Mix Type:           Org_Mix_4_Sounds
Train Samples Cap:  Full dataset
Val Samples Cap:    Full dataset
Loss Objective:     COMPOSITE (STFT Weight: 0.5, Leakage Weight: 0.1)
Enrollment Mode:    2 utterance(s) (VAD: ENABLED)
[Model] Initializing from checkpoint: best_sanada_usef.pt
[Model] Architecture loaded successfully (15,641,885 parameters)
[Memory Optimization] Enabled Gradient Checkpointing (Peak VRAM: ~2.1 GB on 4s chunk)
[Dataset] Looking for SANADA dataset at: /content/drive/.shortcut-targets-by-id/1TxhAg2JEd9vvBxHppDbfGylpUZbW3x7z/SANADA
[Dataset] Mix type: Org_Mix_4_Sounds
[Dataset] Train samples:

---
## Stage C: Curriculum Multi-Speaker Training (2, 4, and 9 Speakers)

- **Dataset**: Combined **15,000 speaker extraction pairs** (2-spk: 2,000 + 4-spk: 4,000 + 9-spk: 9,000)
- **Epochs**: 5
- **Learning Rate**: $2 \times 10^{-5}$
- **Initialization**: Automatically loads `$STAGE_B_SAVE_DIR/best_sanada_usef.pt`
- **Output Directory**: `$DRIVE_EXPERIMENTS_DIR/stage_C_curriculum`


In [ ]:

STAGE_C_SAVE_DIR = f"{DRIVE_EXPERIMENTS_DIR}/stage_C_curriculum"

STAGE_B_BEST = (
    f"{DRIVE_EXPERIMENTS_DIR}/stage_B_4spk/best_sanada_usef.pt"
)

os.makedirs(STAGE_C_SAVE_DIR, exist_ok=True)

# Verify Stage B checkpoint before starting Stage C
if not os.path.isfile(STAGE_B_BEST):
    raise FileNotFoundError(
        f"Stage B checkpoint not found: {STAGE_B_BEST}"
    )

print(f"Stage B checkpoint found: {STAGE_B_BEST}")
print(f"Initializing Stage C training...")
print(f"Stage C output directory: {STAGE_C_SAVE_DIR}")

!python src/train.py \
    --mix-type curriculum \
    --epochs 5 \
    --lr 2e-5 \
    --batch-size 4 \
    --accum-steps 2 \
    --num-workers 2 \
    --device cuda \
    --datasets-dir "$DRIVE_DATASET_DIR" \
    --init-checkpoint "$STAGE_B_BEST" \
    --save-dir "$STAGE_C_SAVE_DIR" \
    --loss-type composite \
    --stft-loss-weight 0.5 \
    --leakage-loss-weight 0.1 \
    --num-enroll-samples 2 \
    --use-vad


Stage B checkpoint found: /content/drive/MyDrive/SANADA_USEF_Experiments/stage_B_4spk/best_sanada_usef.pt
Initializing Stage C training...
Stage C output directory: /content/drive/MyDrive/SANADA_USEF_Experiments/stage_C_curriculum
FINE-TUNING OFFICIAL PRETRAINED USEF-TFGRIDNET ON SANADA
Device:             cuda
Checkpoint Source:  best_sanada_usef.pt
Resume Mode:        False
Epochs:             5
Base Learning Rate: 2e-05
Batch Size:         4 (Effective batch size: 8)
DataLoader Workers: 2
Mix Type:           curriculum
Train Samples Cap:  Full dataset
Val Samples Cap:    Full dataset
Loss Objective:     COMPOSITE (STFT Weight: 0.5, Leakage Weight: 0.1)
Enrollment Mode:    2 utterance(s) (VAD: ENABLED)
[Model] Initializing from checkpoint: best_sanada_usef.pt
[Model] Architecture loaded successfully (15,641,885 parameters)
[Memory Optimization] Enabled Gradient Checkpointing (Peak VRAM: ~2.1 GB on 4s chunk)
[Dataset] Looking for SANADA dataset at: /content/drive/.shortcut-targets-by-

---
## Step 7: Visualize Training Curves & Metrics
Reads `training_log.json` from each completed stage, plots the loss curves and SI-SNRi trajectory, and saves `training_curves.png` to Google Drive.


In [ ]:
import json, os
import matplotlib.pyplot as plt

def plot_training_stage(stage_dir, stage_name):
    log_file = os.path.join(stage_dir, 'training_log.json')
    if not os.path.exists(log_file):
        print(f"No log file found for {stage_name} at: {log_file}")
        return None
    with open(log_file) as f:
        data = json.load(f)
    return data.get('epochs', [])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
stages = [
    (STAGE_A_SAVE_DIR, 'Stage A (2-Spk)', 'tab:blue'),
    (STAGE_B_SAVE_DIR, 'Stage B (4-Spk)', 'tab:orange'),
    (STAGE_C_SAVE_DIR, 'Stage C (Curriculum)', 'tab:green'),
]

for s_dir, s_name, s_color in stages:
    epochs = plot_training_stage(s_dir, s_name)
    if epochs:
        ep_idx = [e['epoch'] for e in epochs]
        train_l = [e['train_loss'] for e in epochs]
        val_l = [e['val_loss'] for e in epochs]
        val_sisnri = [e['val_sisnri_db'] for e in epochs]

        axes[0].plot(ep_idx, train_l, marker='o', label=s_name, color=s_color)
        axes[1].plot(ep_idx, val_l, marker='s', label=s_name, color=s_color)
        axes[2].plot(ep_idx, val_sisnri, marker='^', label=s_name, color=s_color)

axes[0].set_title('Training Loss (Composite)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

axes[2].set_title('Validation SI-SNRi (dB Improvement)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('SI-SNRi (dB)')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend()

plt.tight_layout()
plot_path = os.path.join(DRIVE_EXPERIMENTS_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=300)
print(f"Training curves saved to: {plot_path}")
plt.show()


---
## Step 8: Final Evaluation on the Untouched Held-Out Test Set

Evaluates the trained checkpoints on the untouched test sets:
- **2-Speaker Test Set**: 100 mixtures $\to$ **200 speaker extractions**
- **4-Speaker Test Set**: 100 mixtures $\to$ **400 speaker extractions**

Generates full statistical comparisons against the official baseline (`comparison_baseline_vs_finetuned.json` and `.csv`).


In [ ]:
# Evaluate Stage A Best Checkpoint on 2-Speaker Test Set
!python src/evaluate.py \
    --finetuned-checkpoint "$STAGE_A_SAVE_DIR/best_sanada_usef.pt" \
    --baseline-checkpoint "$INIT_CKPT" \
    --mix-type Org_Mix_2_Sounds \
    --datasets-dir "$DRIVE_DATASET_DIR" \
    --num-samples 0 \
    --use-vad \
    --device cuda


In [ ]:
# Evaluate Stage B Best Checkpoint on 4-Speaker Test Set
!python src/evaluate.py \
    --finetuned-checkpoint "$STAGE_B_SAVE_DIR/best_sanada_usef.pt" \
    --baseline-checkpoint "$INIT_CKPT" \
    --mix-type Org_Mix_4_Sounds \
    --datasets-dir "$DRIVE_DATASET_DIR" \
    --num-samples 0 \
    --use-vad \
    --device cuda


---
## Step 9: Summary Benchmark Report
Displays the evaluation summary comparing Baseline vs Fine-Tuned across the test set.


In [ ]:
import json
import pandas as pd

eval_json = Path('/content/SANADA_final_Project/outputs/evaluation_results/comparison_baseline_vs_finetuned.json')
if eval_json.exists():
    with open(eval_json) as f:
        res = json.load(f)
    print("=" * 70)
    print("BENCHMARK EVALUATION SUMMARY ACROSS UNTOUCHED TEST SET")
    print(f"Total Extractions Evaluated: {res['num_test_samples']}")
    print(f"Baseline Mean SI-SNRi:       {res['baseline']['mean_sisnri_db']:+6.2f} dB (Median: {res['baseline']['median_sisnri_db']:+6.2f} dB)")
    print(f"Fine-Tuned Mean SI-SNRi:     {res['fine_tuned']['mean_sisnri_db']:+6.2f} dB (Median: {res['fine_tuned']['median_sisnri_db']:+6.2f} dB)")
    print(f"Average Delta Gain:          {res['net_improvement']['mean_delta_db']:+6.2f} dB (Max Gain: {res['net_improvement']['max_delta_db']:+6.2f} dB)")
    print("=" * 70)
else:
    print("Run Step 8 evaluation above to generate benchmark metrics.")
